# Example 11.5 / 2: A Minimum-Norm Blend

Companion notebook for **11.5 Second-Order Cones** in the AMPL lab Documentation, section "A Minimum-Norm Blend". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Minimum-Norm Blend

This model finds the smallest Euclidean-norm nonnegative blend that achieves a required score. The objective variable `Radius` and the cone constraint expose the geometry directly.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                # items in the mix
param score {ITEMS} > 0;  # score per unit of each item
param required > 0;       # minimum total score

var Mix {ITEMS} >= 0;  # amount of each item
var Radius >= 0;       # Euclidean norm bound

minimize MinRadius: Radius;  # smallest norm of the mix

subject to Requirement:  # reach the required score
    sum {i in ITEMS} score[i] * Mix[i] >= required;

subject to NormCone:  # second-order cone
    sqrt(sum {i in ITEMS} Mix[i]^2) <= Radius;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["A", "B"]
ampl.param["score"] = {"A": 3, "B": 4}
ampl.param["required"] = 10

In [5]:
# run.py
ampl.solve(solver="gurobi", verbose=False)
print(ampl.solve_result)
print(ampl.var["Mix"].to_dict())
print(round(ampl.var["Radius"].value(), 6))

solved
{'A': 1.200142559417847, 'B': 1.599893095921253}
2.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set ITEMS := A B;
param score := A 3 B 4;
param required := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gurobi;
option solver_msg 0;
solve;
display Mix, Radius;

Mix [*] :=
A  1.20014
B  1.59989
;

Radius = 2



### Another way in amplpy

A driver may pass a native cone (a cone object in the solver's own interface, solved with conic
algorithms), convert it to a quadratic constraint, or leave it as a general nonlinear relation.
Read the solver's log or an exported solver model when native treatment matters.

With the same `model.mod` and `data.py`, Gurobi's own log shows the route. Its lines are captured with `ampl.get_output("solve;")` and
filtered. By default the square-root constraint arrives as a quadratic constraint that Gurobi's
presolve identifies as a second-order cone; with the MP option `cvt:socp=0` the driver does not
convert it, and Gurobi receives a general nonlinear constraint instead. Both runs return
`Radius = 2`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["ITEMS"] = ["A", "B"]
ampl.param["score"] = {"A": 3, "B": 4}
ampl.param["required"] = 10

In [11]:
# run.py
ampl.option["solver"] = "gurobi"
for opts in ("outlev=1", "outlev=1 cvt:socp=0"):
    ampl.option["gurobi_options"] = opts
    log = ampl.get_output("solve;")
    print(opts)
    for line in log.splitlines():
        if "has" in line and "constraint" in line and "objective" not in line:
            print("  ", line.strip())
    print("   Radius =", round(ampl.var["Radius"].value(), 6))

outlev=1
   Model has 1 quadratic constraint
   Presolved model has 1 second-order cone constraint
   Radius = 2.0
outlev=1 cvt:socp=0
   Model has 1 general nonlinear constraint (3 nonlinear terms)
   Presolved model has 2 bilinear constraint(s)
   Presolved model has 1 nonlinear constraint(s)
   Radius = 2.0
